# Notebook 15: K-Fold Ensemble — Binary Classification Evaluation

This notebook evaluates the 5-fold ensemble model for binary DR classification.
It includes per-fold analysis, ROC curves, threshold optimization, and multi-source evaluation.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix,
                             precision_recall_curve, auc as sklearn_auc,
                             brier_score_loss)
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

BASE_DIR = '.'
FIGURES_DIR = f'{BASE_DIR}/figures'
import os
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Setup complete. Loading data...")

## 1. Load APTOS Data

Load ensemble predictions from APTOS test set (531 samples).

In [ ]:
try:
    predictions_df = pd.read_csv(f'{BASE_DIR}/evaluation/aptos_comprehensive/test_predictions_full.csv')
    print(f"Loaded APTOS predictions: {predictions_df.shape[0]} samples")
    print(f"Columns: {predictions_df.columns.tolist()}")
except FileNotFoundError:
    print("ERROR: test_predictions_full.csv not found")
    predictions_df = None

# Extract ensemble columns
if predictions_df is not None:
    y_true = predictions_df['binary_label'].values
    prob_ensemble_no_tta = predictions_df['prob_ensemble_noTTA'].values
    prob_ensemble_tta = predictions_df['prob_ensemble_TTA'].values
    
    # Also extract per-fold probabilities
    prob_fold_1 = predictions_df['prob_fold_1_tta'].values
    prob_fold_2 = predictions_df['prob_fold_2_tta'].values
    prob_fold_3 = predictions_df['prob_fold_3_tta'].values
    prob_fold_4 = predictions_df['prob_fold_4_tta'].values
    prob_fold_5 = predictions_df['prob_fold_5_tta'].values
    
    print(f"\nBinary class distribution:")
    print(predictions_df['binary_label'].value_counts().sort_index())

## 2. Load K-Fold Results

Load per-fold metrics from cross-validation.

In [ ]:
try:
    kfold_results = pd.read_csv(f'{BASE_DIR}/models/kfold/kfold_results.csv')
    print(f"Loaded K-fold results for {len(kfold_results)} folds")
    print("\nPer-fold metrics:")
    print(kfold_results.to_string(index=False))
except FileNotFoundError:
    print("ERROR: kfold_results.csv not found")
    kfold_results = None

## 3. Load Evaluation Metrics

Load comprehensive evaluation metrics including confidence intervals.

In [ ]:
try:
    with open(f'{BASE_DIR}/evaluation/aptos_comprehensive/comprehensive_eval_v2.json', 'r') as f:
        eval_data = json.load(f)
    print("Loaded comprehensive evaluation metrics")
except FileNotFoundError:
    print("ERROR: comprehensive_eval_v2.json not found")
    eval_data = None

## 4. Per-Fold Performance

Visualize metrics across all 5 folds.

## 5. ROC Curves

ROC curves for ensemble variants and individual folds.

In [ ]:
if predictions_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: Ensemble variants
    fpr_no_tta, tpr_no_tta, _ = roc_curve(y_true, prob_ensemble_no_tta)
    auc_no_tta = roc_auc_score(y_true, prob_ensemble_no_tta)
    
    fpr_tta, tpr_tta, _ = roc_curve(y_true, prob_ensemble_tta)
    auc_tta = roc_auc_score(y_true, prob_ensemble_tta)
    
    axes[0].plot(fpr_no_tta, tpr_no_tta, linewidth=2.5, color='#FF9800',
                label=f'Ensemble (no TTA), AUC = {auc_no_tta:.4f}')
    axes[0].plot(fpr_tta, tpr_tta, linewidth=2.5, color='#2196F3',
                label=f'Ensemble (TTA), AUC = {auc_tta:.4f}')
    axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3)
    axes[0].set_xlabel('False Positive Rate', fontsize=11)
    axes[0].set_ylabel('True Positive Rate', fontsize=11)
    axes[0].set_title('ROC Curves: Ensemble Variants', fontsize=12, fontweight='bold')
    axes[0].legend(loc='lower right', fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # Right: Per-fold comparison
    fold_probs = [prob_fold_1, prob_fold_2, prob_fold_3, prob_fold_4, prob_fold_5]
    fold_colors = plt.cm.Set2(np.linspace(0, 1, 5))
    
    for i, (prob, color) in enumerate(zip(fold_probs, fold_colors)):
        fpr, tpr, _ = roc_curve(y_true, prob)
        auc = roc_auc_score(y_true, prob)
        axes[1].plot(fpr, tpr, linewidth=1.5, color=color, alpha=0.7,
                    label=f'Fold {i+1}, AUC = {auc:.4f}')
    
    # Ensemble TTA on top
    axes[1].plot(fpr_tta, tpr_tta, linewidth=2.5, color='black',
                label=f'Ensemble (TTA), AUC = {auc_tta:.4f}')
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3)
    axes[1].set_xlabel('False Positive Rate', fontsize=11)
    axes[1].set_ylabel('True Positive Rate', fontsize=11)
    axes[1].set_title('ROC Curves: Per-Fold Comparison', fontsize=12, fontweight='bold')
    axes[1].legend(loc='lower right', fontsize=9)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/15_roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Ensemble AUC-ROC (no TTA): {auc_no_tta:.6f}")
    print(f"Ensemble AUC-ROC (TTA): {auc_tta:.6f}")

## 6. Precision-Recall Curves

PR curves for ensemble variants.

## 7. Confusion Matrices

2x2 confusion matrices for ensemble variants.

## 8. Threshold Analysis

Load and analyze threshold sweeps for ensemble.

## 9. Metrics Summary Table

Comprehensive metrics comparison.